# Intra-Benchmark Baseline Comparisons

This notebook builds non-forecaster baseline predictions for the Lyptus task outcomes used by `intra_benchmark_calibration`, using the reusable helpers in `calculate_baselines.py`.

Baselines implemented by the script:

- `uninformed_0_5`: predict 0.5 for every `(model, task)` cell.
- `model_pass_rate`: predict each model's average pass rate across all evaluated headline tasks.
- `model_bin_pass_rate`: predict each model's average pass rate within the target task's difficulty bin.
- `irt_logistic_fit`: predict from the Lyptus per-model logistic fit at the target task's FST minutes.

The default evaluation panel samples 10 target tasks from each difficulty bin, then scores every forecasted model on those tasks when a ground-truth outcome is available.

This notebook is primarily for evaluating the performance of the baselines in general. There is also a cell at the end for evaluating the baselines against a specific elicitation run.


In [ ]:
from __future__ import annotations

from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)
pd.set_option("display.max_colwidth", 120)
sns.set_theme(style="whitegrid")


def find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "intra_benchmark_calibration").exists():
            return candidate
    raise FileNotFoundError("Could not find repo root containing intra_benchmark_calibration")


REPO_ROOT = find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from report_analyses.calculate_baselines import (
    BaselineConfig,
    add_baseline_predictions,
    build_baseline_context,
    calculate_baselines,
    default_run_label,
    load_run_frames,
    make_evaluation_panel,
    sample_targets_by_bin,
    score_baselines,
    score_elicited_predictions,
)

REPO_ROOT


## Configuration

These defaults mirror `config_full.yaml`: 5 equal-count bins, `/home/jeffm/lyptus-data`, and the sparse older models dropped from the outcome panel.

In [ ]:
LYPTUS_REPO_DIR = Path("/home/jeffm/lyptus-data")
DROP_MODELS = ("GPT-2", "GPT-3", "GPT-3.5")

# Use "all" or any subset of: uninformed_0_5, model_pass_rate, model_bin_pass_rate, irt_logistic_fit.
BASELINES = "all"

N_BINS = 5
BINNING_STRATEGY = "equal_count"
EXPLICIT_EDGES = None

N_TARGETS_PER_BIN = 10
TARGET_SAMPLE_SEED = 20260514

BOOTSTRAP_ITERATIONS = 1000
BOOTSTRAP_SEED = 20260515
METRIC_BOOTSTRAP_QUANTILES = (0.025, 0.975)

IRT_BOOTSTRAP_ITERATIONS = 250
IRT_BOOTSTRAP_SEED = BOOTSTRAP_SEED + 1000

BASELINE_CONFIG = BaselineConfig(
    lyptus_repo_dir=LYPTUS_REPO_DIR,
    drop_models=DROP_MODELS,
    n_bins=N_BINS,
    binning_strategy=BINNING_STRATEGY,
    explicit_edges=EXPLICIT_EDGES,
    bootstrap_iterations=BOOTSTRAP_ITERATIONS,
    bootstrap_seed=BOOTSTRAP_SEED,
    metric_bootstrap_quantiles=METRIC_BOOTSTRAP_QUANTILES,
    irt_bootstrap_iterations=IRT_BOOTSTRAP_ITERATIONS,
    irt_bootstrap_seed=IRT_BOOTSTRAP_SEED,
)

# Optional: set to a specific run directory to compare against actual elicitation rows.
RUN_DIR = REPO_ROOT / "intra_benchmark_calibration" / "experiments" / "G_model_sweep" / "results" / "gpt55"


## Load Lyptus Outcomes, Difficulty Bins, And IRT Fits

In [ ]:
baseline_context = build_baseline_context(BASELINE_CONFIG, baselines=BASELINES)

dataset = baseline_context.dataset
bins = baseline_context.bins
tasks = baseline_context.tasks
irt_fit_data = baseline_context.irt_fit_data
irt_fit_params = baseline_context.irt_fit_params
irt_bootstrap_fit_params = baseline_context.irt_bootstrap_fit_params

if not irt_bootstrap_fit_params.empty:
    display(
        irt_bootstrap_fit_params.groupby("agent")
        .agg(n_bootstrap=("bootstrap_idx", "size"), success_rate=("fit_success", "mean"))
        .loc[[model for model in dataset.outcomes.models if model in irt_bootstrap_fit_params["agent"].unique()]]
    )


## Sample Target Tasks

For the first comparator panel, sample 10 random target tasks from each difficulty bin. The seed makes the sample reproducible.

In [ ]:
sampled_targets = sample_targets_by_bin(
    tasks,
    n_targets_per_bin=N_TARGETS_PER_BIN,
    seed=TARGET_SAMPLE_SEED,
)
# display(sampled_targets.groupby("target_bin").size().rename("n_sampled"))
# display(sampled_targets)


## Build Baseline Predictions

In [ ]:
overall_scores, panel = calculate_baselines(
    panel=make_evaluation_panel(sampled_targets, baseline_context),
    context=baseline_context,
    baselines=BASELINES,
    return_panel=True,
)
# display(panel.head())
# display(
#     panel.groupby(["target_bin", "forecasted_model"])
#     .agg(
#         n=("outcome", "size"),
#         empirical_pass_rate=("outcome", "mean"),
#         p_model_pass_rate=("p_model_pass_rate", "mean"),
#         p25_model_pass_rate=("p25_model_pass_rate", "mean"),
#         p75_model_pass_rate=("p75_model_pass_rate", "mean"),
#         p_model_bin_pass_rate=("p_model_bin_pass_rate", "mean"),
#         p25_model_bin_pass_rate=("p25_model_bin_pass_rate", "mean"),
#         p75_model_bin_pass_rate=("p75_model_bin_pass_rate", "mean"),
#         p_irt_logistic_fit=("p_irt_logistic_fit", "mean"),
#         p25_irt_logistic_fit=("p25_irt_logistic_fit", "mean"),
#         p75_irt_logistic_fit=("p75_irt_logistic_fit", "mean"),
#     )
#     .reset_index()
# )
# display(panel.groupby("target_bin").agg(n=("outcome", "size"), empirical_pass_rate=("outcome", "mean")))


## Score Baselines

`brier` is the main comparator for point predictions. `crps_beta` uses the same Beta-fit CRPS machinery as `analyse_results.py` when a baseline has `p25/p50/p75` quantiles. The pass-rate intervals come from empirical bootstraps, and the IRT interval comes from bootstrapped logistic refits.

The score intervals below are non-parametric row bootstraps over the evaluation panel cells within each reported group.


In [ ]:
overall_scores = overall_scores.sort_values("brier")
display(overall_scores)

by_bin_scores = score_baselines(
    panel,
    ["target_bin"],
    baselines=BASELINES,
    config=BASELINE_CONFIG,
    seed_offset=1,
).sort_values(["target_bin", "brier"])
display(by_bin_scores)

by_model_scores = score_baselines(
    panel,
    ["forecasted_model"],
    baselines=BASELINES,
    config=BASELINE_CONFIG,
    seed_offset=2,
).sort_values(["baseline", "brier"])
display(by_model_scores)


In [ ]:
def add_errorbars(ax, data: pd.DataFrame, x: str, y: str, low: str, high: str, *, color: str = "black") -> None:
    for x_value, row in enumerate(data.itertuples(index=False)):
        y_value = getattr(row, y)
        low_value = getattr(row, low)
        high_value = getattr(row, high)
        if pd.isna(y_value) or pd.isna(low_value) or pd.isna(high_value):
            continue
        ax.errorbar(
            x=x_value,
            y=y_value,
            yerr=[[y_value - low_value], [high_value - y_value]],
            fmt="none",
            ecolor=color,
            elinewidth=1.5,
            capsize=4,
            capthick=1.5,
            zorder=3,
        )


def plot_metric_by_bin(scores: pd.DataFrame, metric: str, ylabel: str, title: str) -> None:
    fig, ax = plt.subplots(figsize=(8, 4))
    for baseline_name, baseline_scores in scores.sort_values("target_bin").groupby("baseline", sort=False):
        ax.errorbar(
            baseline_scores["target_bin"],
            baseline_scores[metric],
            yerr=[
                baseline_scores[metric] - baseline_scores[f"{metric}_ci_low"],
                baseline_scores[f"{metric}_ci_high"] - baseline_scores[metric],
            ],
            marker="o",
            capsize=3,
            linewidth=1.5,
            label=baseline_name,
        )
    ax.set_xticks(range(N_BINS))
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.legend(loc="center left", bbox_to_anchor=(1.02, 0.5))
    plt.show()


fig, ax = plt.subplots(figsize=(7, 4))
sns.barplot(data=overall_scores, x="baseline", y="brier", ax=ax, color="steelblue", edgecolor="black")
add_errorbars(ax, overall_scores.reset_index(drop=True), "baseline", "brier", "brier_ci_low", "brier_ci_high")
ax.axhline(0.25, color="grey", linestyle="--", linewidth=1)
ax.set_xlabel("")
ax.set_ylabel("Brier score")
ax.set_title("Baseline Brier Scores On Sampled Target Panel")
ax.tick_params(axis="x", rotation=20)
plt.show()

crps_overall = overall_scores.dropna(subset=["crps_beta"]).sort_values("crps_beta").reset_index(drop=True)
fig, ax = plt.subplots(figsize=(7, 4))
sns.barplot(data=crps_overall, x="baseline", y="crps_beta", ax=ax, color="darkseagreen", edgecolor="black")
add_errorbars(ax, crps_overall, "baseline", "crps_beta", "crps_beta_ci_low", "crps_beta_ci_high")
ax.set_xlabel("")
ax.set_ylabel("Beta CRPS")
ax.set_title("Baseline Beta CRPS On Sampled Target Panel")
ax.tick_params(axis="x", rotation=20)
plt.show()

plot_metric_by_bin(by_bin_scores, "brier", "Brier score", "Baseline Brier By Target Difficulty Bin")
plot_metric_by_bin(
    by_bin_scores.dropna(subset=["crps_beta"]),
    "crps_beta",
    "Beta CRPS",
    "Baseline Beta CRPS By Target Difficulty Bin",
)


## Optional: Score Baselines On An Existing Calibration Run

This section evaluates the same baselines on the exact `(forecasted_model, target_task_id, target_bin, outcome)` cells from a run CSV, then compares them with the elicited `p50` scores. This is the most direct apples-to-apples comparator for a completed `intra_benchmark_calibration` run.

In [ ]:
def run_panel_from_csv(run_dir: Path) -> pd.DataFrame:
    _raw, scoreable = load_run_frames(run_dir, default_run_label(run_dir))
    cols = [
        "forecasted_model",
        "target_task_id",
        "target_task_family",
        "target_fst_minutes",
        "target_bin",
        "outcome",
        "p25",
        "p50",
        "p75",
    ]
    cols = [c for c in cols if c in scoreable.columns]
    return scoreable[cols].drop_duplicates(["forecasted_model", "target_task_id", "target_bin", "outcome"]).reset_index(drop=True)


if RUN_DIR.exists():
    run_panel = run_panel_from_csv(RUN_DIR)
    run_panel = add_baseline_predictions(run_panel, baseline_context, baselines=BASELINES)
    run_scores = score_baselines(run_panel, baselines=BASELINES, config=BASELINE_CONFIG, seed_offset=3)

    elicited_score = score_elicited_predictions(
        run_panel.dropna(subset=["p50"]),
        config=BASELINE_CONFIG,
        seed_offset=4,
    ).rename(columns={"source": "baseline"}).drop(columns=["source_type"])

    display(pd.concat([run_scores, elicited_score], ignore_index=True).sort_values("brier"))
else:
    print(f"RUN_DIR does not exist: {RUN_DIR}")
